# Laboratorio de regresión - 5

|                |   |
:----------------|---|
| **Nombre**     Valeria Estefanía Milke Loera   |
| **Fecha**       16-02-2026|   |
| **Expediente** 739228 |   |

## Validación

Hemos estado usando `train_test_split` en nuestros modelos anteriores.

¿Por qué?

Porque nos permite evaluar el desempeño real de nuestro modelo. Cuando entrenamos un modelo con todos los datos y luego lo evaluamos con esos mismos datos, el resultado puede ser engañoso, ya que el modelo podría estar repitiendo la información en lugar de aprender los patrones generales. Al dividir el conjunto de datos en una parte de entrenamiento y otra de prueba, simulamos una situación más realista en la que el modelo debe hacer predicciones sobre datos que no ha visto antes. De esta forma, podemos medir su capacidad de generalización.

Si la muestra es un subset de la población y queremos generalizar sobre la población, ¿no sería mejor utilizar todos los datos al entrenar un modelo?

Hacer eso desde un inicio podría crear un modelo que aprenda a hacer una regresión acertada de los datos ya existentes pero que a la hora de predecir nuevos datos se cree un sobreajuste. Separando las muestra en una de entrenamiento y otra de prueba, podemos verificar qué tanto generaliza el modelo y qué tan bien funciona con datos nuevos.  

El propósito de volver a muestrear dentro de nuestro dataset es tener una idea de qué tan buena podría ser la generalización de nuestro modelo. Imagina un dataset ya separado en dos mitades. Utilizas la primera mitad para entrenar el modelo y pruebas en la segunda mitad; la segunda mitad eran datos invisibles para el modelo al momento de entrenar. Esto nos lleva a tres escenario típicos:

1. Si el modelo hace buenas predicciones en la segunda mitad, significa que la primera mitad era "suficiente" para generalizar.
2. Si el modelo no hace buenas predicciones en la segunda mitad, pero sí en la primera mitad, podría ser que había información importante en la segunda mitad que debió haber sido tomada en cuenta al entrenar, o un problema de overfitting.
3. Si el modelo no hace buenas predicciones en la segunda mitad, y tampoco en la primera mitad, se tendrían que revisar los factores y/o el modelo seleccionado.

El caso ideal sería el 1, pero por estadística los errores y varianzas tienen como entrada el número de muestas, por lo que tenemos menos seguridad de nuestros resutados al usar menos muestras. Si vemos que el modelo generaliza bien podemos unir de nuevo el dataset y entrenar sobre el dataset completo.

En el caso 2 está el problema de que no podemos saber qué información es necesaria para el entrenamiento apropiado del modelo; esto nos lleva a pensar que debemos usar el dataset completo para entrenar, pero esto nos lleva al mismo problema de no saber si el modelo puede generalizar.

El problema sólo incrementa si se tienen hiperparámetros en el modelo (e.g. $\lambda$ en regularización).

## Leave-One-Out Cross Validation

Este método de validación es una colección de $n$ `train-test-split`. Teniendo un dataset de $n$ muestras, la lógica es:
1. Saca una muestra del dataset.
2. Entrena tu modelo con las $n-1$ muestras.
3. Evalúa tu modelo en la muestra que quedó fuera con el métrico que más se ajuste a la aplicación.
4. Regresa la muestra al dataset.
5. Repite 1-4 con muestras diferentes hasta haber hecho el procedimiento $n$ veces para $n$ muestras.
6. Calcula la media y desviación estándar de los métricos guardados.

Con los resultados del proceso de validación podemos saber qué tan bueno podría ser el modelo seleccionado con los datos (con/sin transformaciones).

### Ejercicio 1

Utiliza el dataset `Motor Trend Car Road Tests`. Elimina la columna `model` y entrena 32 modelos diferentes utilizando Leave-One-Out Cross Validation con target `mpg`. Utiliza MSE como métrico.

In [18]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LinearRegression
from scipy import stats
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [19]:
df0 = pd.read_excel(r"C:\Users\valer\Motor Trend Car Road Tests.xlsx")
df = df0.drop(columns=["model"])
df.head()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [20]:
x = df.drop(columns=["mpg"]).values
y = df["mpg"].values

In [23]:
n = len(df)
lr = LinearRegression()
msel = []

In [27]:
for i in range(n):
    xte = x[i].reshape(1, -1)
    yte = y[i].reshape(1,)
    xtr = np.delete(x, i, axis=0)
    ytr = np.delete(y, i)
    lr.fit(xtr, ytr)
    ypred = lr.predict(xte)
    mse = mean_squared_error(yte, ypred)
    msel.append(mse)
print("MSE promedio:", np.mean(msel))
print("Desviación estándar:",np.std(msel))

MSE promedio: 12.18155800690196
Desviación estándar: 17.06739987188855


Interpreta.

El valor de MSE promedio = 12.18 representa el error cuadrático medio. Esto significa que, el modelo comete un error cuadrático de aproximadamente 12.18 unidades al predecir el valor de mpg cuando la observación evaluada no fue utilizada en el entrenamiento. En otras palabras, esta es una estimación del error fuera de muestra, es decir, del desempeño esperado del modelo ante datos nuevos. La raíz de 12.18 es aproximadamente 3.49, lo que indica que el modelo se equivoca, en promedio, alrededor de 3.5 millas por galón en sus predicciones.

Por otro lado, la desviación estándar del MSE = 17.07 indica que el error no es completamente estable entre las distintas iteraciones del LOOCV. Esto significa que algunas observaciones generan errores mucho mayores que otras cuando se dejan fuera del entrenamiento. En términos prácticos, el modelo no predice con la misma precisión todos los vehículos; existen ciertos casos donde la predicción es considerablemente menos precisa.

En conjunto, estos resultados sugieren que el modelo tiene una capacidad de generalización moderada: logra predecir el consumo de combustible con un error promedio razonable, pero presenta variabilidad en su desempeño dependiendo de la observación evaluada.

## K-Folds Cross-Validation

El dataset `Motor Trend Car Road Tests` sólo tiene 32 muestras, y utilizar un modelo sencillo de regresión múltiple hace que usar LOOCV sea muy rápido. El dataset `California Housing` tiene $20640$ muestras para $9$ columnas, entonces realizar un ajuste sobre una transformación o sobre el modelo y luego calcular el impacto esperado podría tomar más tiempo.

La solución propuesta es dividir el dataset en *k* folds (partes iguales), ajustar en *k-1* folds y probar en el restante.

### Ejercicio 2
Utiliza el dataset `California Housing` y haz K-folds Cross Validation con 10 folds. Utiliza el MSE como métrico.

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print("Dataset Shape:", housing.data.shape, housing.target.shape)
print("Dataset Features:", housing.feature_names)
print("Dataset Target:", housing.target_names)
X = housing.data
y = housing.target

Interpreta.

## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3